# AR Family Event 2020

In [1]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
import os

import matplotlib.path as mpath
import matplotlib.colors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.util import add_cyclic_point

from matplotlib import animation
import matplotlib
matplotlib.use("Agg")
from IPython.display import Video
from matplotlib.cm import prism

from ipywidgets import IntProgress
from IPython.display import display

repo_dir = str(Path(os.getcwd()).parents[0])
os.chdir(repo_dir + '/scripts/')
from utils import display_catalog

In [2]:
repo_dir = str(Path(os.getcwd()).parents[0])
os.chdir(repo_dir + '/notebooks/')

catalog = pd.read_pickle('storm_df_epstime_12.pkl')

def compute_duration(ar_da):
    days = (ar_da.time.max() - ar_da.time.min()).values.astype('timedelta64[h]').astype(int) + np.timedelta64(3, 'h')
    return days

def add_start_date(ar_da):
    start = ar_da.time.min().values
    return start

def add_end_date(ar_da):
    end = ar_da.time.max().values
    return end

In [3]:
catalog['duration'] = catalog['data_array'].apply(compute_duration)
catalog['start_date'] = catalog['data_array'].apply(add_start_date)
catalog['end_date'] = catalog['data_array'].apply(add_end_date)

In [4]:
ars_2020 = catalog[catalog.end_date.dt.year == 2020]

In [5]:
feb_ars = ars_2020[ars_2020.end_date.dt.month == 2]

In [6]:
labels = []
times = []
lats = []
lons = []

for index in feb_ars.index:

    label = index
    storm_da = feb_ars.loc[index].data_array
    storm_times = storm_da.time

    for i in range(len(storm_times)):
        time_slice = storm_da.sel(time = storm_times[i])
        inds = np.argwhere(time_slice.to_numpy() == 1)
        storm_lats = time_slice.lats[inds[:,0]].values
        storm_lons = time_slice.lons[inds[:,1]].values
        labels.append(label)
        times.append(storm_times[i].values)
        lats.append(storm_lats)
        lons.append(storm_lons)

In [7]:
cluster_time_df = pd.DataFrame({'label':labels, 'time':times, 'lat':lats, 'lon':lons})

In [8]:
times = np.unique(cluster_time_df.time)

unique_clusters = cluster_time_df['label'].unique()
color_mapping = {unique_clusters[j]:prism(j/len(unique_clusters)) for j in range(len(unique_clusters)) }


def plt_time(j):
    time_pt = times[j]

    dat = cluster_time_df[cluster_time_df['time'] == time_pt]
    n_clusts = dat.shape[0]

    for i in range(n_clusts):
        cluster = dat['label'].iloc[i]
        ax.scatter(dat['lon'].iloc[i], dat['lat'].iloc[i], transform=ccrs.PlateCarree(), s=1, color=color_mapping[cluster], label=str(cluster), zorder=30)
        
    ax.legend()

    ax.set_extent([-180,180,-90,-39], ccrs.PlateCarree())
    #land_50m = cfeature.NaturalEarthFeature('physical', 'land', '50m',edgecolor='none',facecolor='white') # 10m, 50m, 110m
    #ax.add_feature(land_50m,linewidth=3)
    ice_shelf_poly = cfeature.NaturalEarthFeature('physical', 'antarctic_ice_shelves_polys', '50m',edgecolor='none',facecolor='lightcyan') # 10m, 50m, 110m
    ax.add_feature(ice_shelf_poly,linewidth=3)
    ice_shelf_line = cfeature.NaturalEarthFeature('physical', 'antarctic_ice_shelves_lines', '50m',edgecolor='black',facecolor='none') # 10m, 50m, 110m
    ax.add_feature(ice_shelf_line,linewidth=1,zorder=13)
    ax.coastlines(resolution='110m',linewidth=1,zorder=32)

    
    # Map extent 
    theta = np.linspace(0, 2*np.pi, 100)
    center, radius = [0.5, 0.5], 0.5
    verts = np.vstack([np.sin(theta), np.cos(theta)]).T
    circle = mpath.Path(verts * radius + center)
    ax.set_boundary(circle, transform=ax.transAxes)
    ax.gridlines(alpha=0.5, zorder=33)
    
    time_ts = pd.Timestamp(time_pt)
    ax.set_title(f'{time_ts.month}/{time_ts.day}/{time_ts.year}, {time_ts.hour}:00')




In [9]:
fig, ax = plt.subplots(figsize=(5,5), subplot_kw=dict(projection=ccrs.Stereographic(central_longitude=0., central_latitude=-90.)))
plt_time(0)
fig.suptitle('Eps Time = 12 hours', fontsize=16)

def update_img(i):
    ax.clear()
    plt_time(i)

ani = animation.FuncAnimation(fig, update_img, frames=len(times))

In [10]:
def save_progress(i, n):
    if i%5 == 0: {print(f'Saving frame {i}/{n}')}

ani.save('2020_family_event.mp4', progress_callback=save_progress)

Saving frame 0/133


MovieWriter stderr:
Unknown encoder 'h264'



CalledProcessError: Command '['ffmpeg', '-f', 'rawvideo', '-vcodec', 'rawvideo', '-s', '500x500', '-pix_fmt', 'rgba', '-framerate', '5.0', '-loglevel', 'error', '-i', 'pipe:', '-vcodec', 'h264', '-pix_fmt', 'yuv420p', '-y', '2020_family_event.mp4']' returned non-zero exit status 1.

In [14]:
Video('2020_family_event.mp4')